In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    os.getenv("DATASET_DB_URL", "mysql+pymysql://root:@127.0.0.1:3307/family_finance_mono?charset=utf8mb4")
)

In [8]:
# Gastos e ingresos
df = pd.read_sql("""
SELECT
   combined.year_month,
   BIN_TO_UUID(combined.family_id_bin) AS family_id,
   BIN_TO_UUID(combined.category_id_bin) AS category_id,
   SUM(combined.total_income) AS total_income,
   SUM(combined.total_expenses) AS total_expenses
FROM (
   SELECT
      DATE_FORMAT(ie.date, '%%Y-%%m') AS `year_month`,
      fm.family_id AS family_id_bin,
      ie.category_id AS category_id_bin,
      SUM(ie.amount) AS total_income,
      0 AS total_expenses
   FROM income_entries ie
   LEFT JOIN family_members fm ON ie.family_member_id = fm.id
   WHERE ie.is_active = 1
   GROUP BY
      DATE_FORMAT(ie.date, '%%Y-%%m'),
      fm.family_id,
      ie.category_id

   UNION ALL

   SELECT
      DATE_FORMAT(e.date, '%%Y-%%m') AS `year_month`,
      fm.family_id AS family_id_bin,
      e.category_id AS category_id_bin,
      0 AS total_income,
      SUM(e.amount) AS total_expenses
   FROM expenses e
   LEFT JOIN family_members fm ON e.family_member_id = fm.id
   GROUP BY
      DATE_FORMAT(e.date, '%%Y-%%m'),
      fm.family_id,
      e.category_id
) AS combined
GROUP BY
   combined.year_month,
   combined.family_id_bin,
   combined.category_id_bin
ORDER BY
   combined.year_month,
   combined.family_id_bin,
   combined.category_id_bin
""", engine)



In [9]:
# Sacamos el balance
df['balance'] = df['total_income'] - df['total_expenses']

In [10]:
df.head(10)

,year_month,family_id,category_id,total_income,total_expenses,balance
0,2022-01,b006e9da-0024-426a-8f7d-dfa84d676ff7,7ff3c9d0-805a-447f-9ee3-061b8d749c8c,0.00,346.81,-346.81
1,2022-01,d0000000-0000-0000-0000-000000000000,10d59f27-c95c-450e-a6fd-c3430059ce4b,0.00,1881.88,-1881.88
2,2022-01,d0000000-0000-0000-0000-000000000000,8ac5c50f-69ed-46d2-a886-0651726336c0,1261.72,0.00,1261.72
3,2022-01,d0000000-0000-0000-0000-000000000000,a214144b-3db3-48a3-8233-5a2d00beecff,0.00,276.30,-276.30
4,2022-01,d0000000-0000-0000-0000-000000000000,d4272c6d-5bf1-4404-b8f0-9e54cd9ef425,0.00,309.20,-309.20
5,2022-01,d1111111-1111-1111-1111-111111111111,10d59f27-c95c-450e-a6fd-c3430059ce4b,0.00,4461.99,-4461.99
6,2022-01,d1111111-1111-1111-1111-111111111111,3527e48f-9793-45bc-a402-9a68654b211a,15006.92,130.18,14876.74
7,2022-01,d1111111-1111-1111-1111-111111111111,8ac5c50f-69ed-46d2-a886-0651726336c0,0.00,2521.39,-2521.39
8,2022-01,d2222222-2222-2222-2222-222222222222,3527e48f-9793-45bc-a402-9a68654b211a,0.00,729.56,-729.56
9,2022-01,d2222222-2222-2222-2222-222222222222,5eb37d55-8ff2-4584-b5e2-7e0e4b2ddada,2528.79,0.00,2528.79


In [ ]:
# Extraemos los datos a un csv
df.to_csv('../../data/family-finance-category-data.csv', index=False)